# `uv` -- A Fast Python Package Manager

[`uv`](https://docs.astral.sh/uv/) is a modern Python package and project manager written in Rust. It is a drop-in replacement for `pip` + `venv` + `pip-tools` + `pipx` + parts of `poetry`, and it is **10 to 100 times faster** than traditional tools.

This notebook walks you through the basics. We demonstrate commands by calling `uv` from the notebook with the `!` shell-execution prefix.

## 1. Installation

On Windows (PowerShell):
```powershell
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

On macOS / Linux:
```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

Or via pip / conda:
```bash
pip install uv
conda install -c conda-forge uv
```

Check that it worked:

In [ ]:
!uv --version

## 2. Create a playground directory

All our experiments go into a temporary folder so we don't affect anything else.

In [ ]:
import os, shutil

PLAYGROUND = "uv_playground"
if os.path.exists(PLAYGROUND):
    shutil.rmtree(PLAYGROUND)
os.makedirs(PLAYGROUND)
os.chdir(PLAYGROUND)
print("Working in:", os.getcwd())

## 3. Virtual environments -- `uv venv`

A virtual environment is an isolated Python installation. `uv venv` creates one in **a fraction of a second** (compared to several seconds with `python -m venv`).

In [ ]:
!uv venv myenv

This creates a `myenv/` folder with an isolated Python. To activate it outside this notebook:
- Windows: `myenv\Scripts\activate`
- macOS / Linux: `source myenv/bin/activate`

## 4. Installing packages -- `uv pip install`

`uv pip` mirrors the `pip` command-line interface, so most `pip` commands work if you just replace `pip` with `uv pip`:

In [ ]:
# Install into the virtual environment we just created
!uv pip install --python myenv requests rich

In [ ]:
# List installed packages
!uv pip list --python myenv

Notice how fast that was. `uv` downloads packages in parallel and caches them globally, so future installs of the same package are practically instant.

## 5. Projects -- `uv init`

For real projects, `uv` supports a modern `pyproject.toml`-based workflow. Let's create a small project:

In [ ]:
!uv init my_project

In [ ]:
# See what was generated
import os
for f in sorted(os.listdir("my_project")):
    print(f)

In [ ]:
# Show the pyproject.toml file
with open("my_project/pyproject.toml") as f:
    print(f.read())

## 6. Adding dependencies -- `uv add`

`uv add` installs a package and records it in `pyproject.toml` automatically.

In [ ]:
os.chdir("my_project")

!uv add numpy pandas

In [ ]:
# pyproject.toml was updated
with open("pyproject.toml") as f:
    print(f.read())

A file called `uv.lock` was also generated. This **lockfile** pins the exact versions of every dependency (and its sub-dependencies). Commit it to git so your colleagues get the same versions.

In [ ]:
# First few lines of the lockfile
with open("uv.lock") as f:
    print("".join(f.readlines()[:25]))

## 7. Running code -- `uv run`

`uv run` executes a Python script using the project's environment. It **auto-installs** missing dependencies first, so you never forget to activate the venv.

In [ ]:
# Write a small script
script = '''
import numpy as np
import pandas as pd

df = pd.DataFrame({"x": np.arange(5), "y": np.arange(5) ** 2})
print(df)
'''
with open("demo.py", "w") as f:
    f.write(script)

!uv run demo.py

## 8. Syncing an existing project -- `uv sync`

When you clone a `uv` project from git, run `uv sync` once. It reads `uv.lock` and sets up an identical environment in seconds.

In [ ]:
!uv sync

## 9. Inline script dependencies (bonus)

A great feature for one-off scripts: declare dependencies **inside the script itself** using a special comment. Then `uv run` creates a disposable environment on the fly.

In [ ]:
inline_script = '''# /// script
# requires-python = ">=3.10"
# dependencies = ["rich"]
# ///
from rich import print
print("[bold magenta]Hello from a self-contained script![/bold magenta]")
'''
with open("hello.py", "w") as f:
    f.write(inline_script)

!uv run hello.py

You can share `hello.py` with anyone. All they need is `uv` -- no manual `pip install`, no virtual environment setup.

## 10. Using uv with Jupyter notebooks

To use a `uv` project's environment as a Jupyter kernel, you need two things:
1. Add **`ipykernel`** as a development dependency
2. Register the project's `.venv` as a Jupyter kernel

Then VS Code, JupyterLab, or classic Jupyter will see it in the kernel picker.

In [ ]:
# Still inside my_project/
!uv add --dev ipykernel

`--dev` marks it as a development dependency -- it is saved under a `[dependency-groups.dev]` table in `pyproject.toml` instead of the main dependencies, so production users don't have to install it.

Now register the kernel. The `--env VIRTUAL_ENV <path>` flag tells Jupyter which virtualenv the kernel belongs to, so Python imports resolve to the project's packages:

In [ ]:
# Compute the absolute path to the .venv (cross-platform -- no $(pwd) needed)
venv_path = os.path.abspath(".venv")
print("VIRTUAL_ENV =", venv_path)

!uv run ipython kernel install --user --env VIRTUAL_ENV "{venv_path}" --name=project

> On macOS/Linux with bash you can also write it in the shorter form shown in the `uv` docs:
> ```bash
> uv run ipython kernel install --user --env VIRTUAL_ENV $(pwd)/.venv --name=project
> ```
> The Python-variable version above works on Windows too.

Verify the kernel is registered:

In [ ]:
!jupyter kernelspec list

Now in VS Code / Jupyter, select the kernel named **`project`** (or whatever `--name=` you chose) and your notebook runs inside the `uv` project's virtualenv.

To remove a kernel later:
```bash
jupyter kernelspec remove project
```

## 11. Cheat sheet

| Task | Command |
|------|---------|
| Install uv | `irm https://astral.sh/uv/install.ps1 \| iex` (Windows) |
| Create a virtual env | `uv venv` |
| Install a package | `uv pip install <pkg>` |
| Start a new project | `uv init my_project` |
| Add a dependency | `uv add <pkg>` |
| Add a dev dependency | `uv add --dev <pkg>` |
| Remove a dependency | `uv remove <pkg>` |
| Run a script | `uv run script.py` |
| Sync from lockfile | `uv sync` |
| Upgrade lockfile | `uv lock --upgrade` |
| Register Jupyter kernel | `uv run ipython kernel install --user --name=project` |
| Install a CLI tool globally | `uv tool install <pkg>` |
| Run a tool once | `uv tool run <pkg>` (alias: `uvx`) |

**Docs:** https://docs.astral.sh/uv/

## 12. Cleanup

Remove the playground folder and the demo kernel when you are done experimenting:

In [ ]:
# Remove the demo kernel
!jupyter kernelspec remove -f project

# Remove the playground folder
os.chdir("../..")
shutil.rmtree(PLAYGROUND, ignore_errors=True)
print("Cleaned up.")